# 1. Limpieza y preparación de datos

**Objetivo:** Leer el archivo DATALOG.TXT con formato de texto libre, extraer las mediciones de temperatura y humedad de los tres sensores (Casa Guadua, Casa Plástico, Ambiente) y estructurarlos en un DataFrame. Además, limpiar valores erróneos (-999).

In [ ]:
## 1.1 Importar librerías
import re
import pandas as pd
import numpy as np

1.2 Leer el archivo raw


In [31]:
from google.colab import drive
drive.mount('/content/drive')

# Assuming 'DATALOG.txt' is directly in the 'INGE AMBIENTAL' folder in your Google Drive
ruta_raw = '/content/drive/MyDrive/INGE AMBIENTAL/DATALOG.TXT'
print(f"Attempting to open file at path: {ruta_raw}")
with open(ruta_raw, "r", encoding="utf-8") as f:
    lineas = f.readlines()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Attempting to open file at path: /content/drive/MyDrive/INGE AMBIENTAL/DATALOG.TXT


1.3 Parsear las líneas para extraer cada muestra


In [32]:
muestras = []
muestra_actual = {}
for linea in lineas:
    linea = linea.strip()
    if linea.startswith("Muestra #"):
        num = int(re.search(r"\d+", linea).group())
        muestra_actual["muestra"] = num
    elif linea.startswith("S1 Casa Guadua:"):
        valores = re.findall(r"([\d\.\-]+)", linea)
        if len(valores) >= 2:
            muestra_actual["T_guadua"] = float(valores[0])
            muestra_actual["H_guadua"] = float(valores[1])
    elif linea.startswith("S2 Casa Plastico:"):
        valores = re.findall(r"([\d\.\-]+)", linea)
        if len(valores) >= 2:
            muestra_actual["T_plastico"] = float(valores[0])
            muestra_actual["H_plastico"] = float(valores[1])
    elif linea.startswith("S4 Ambiente :"):
        valores = re.findall(r"([\d\.\-]+)", linea)
        if len(valores) >= 2:
            muestra_actual["T_ambiente"] = float(valores[0])
            muestra_actual["H_ambiente"] = float(valores[1])
    elif linea.startswith("--------------------------------"):
        if muestra_actual:
            muestras.append(muestra_actual)
            muestra_actual = {}

1.4 Crear DataFrame y limpiar


In [33]:
df = pd.DataFrame(muestras)
# Eliminar filas con algún valor -999 (error de sensor)
df_clean = df[(df != -999).all(axis=1)].copy()
# Agregar tiempo en minutos (muestra 1 -> tiempo 0)
df_clean["tiempo_min"] = df_clean["muestra"] - 1
# Reordenar columnas
columnas = ["muestra", "tiempo_min", "T_guadua", "H_guadua", "T_plastico", "H_plastico", "T_ambiente", "H_ambiente"]
df_clean = df_clean[columnas]
df_clean.reset_index(drop=True, inplace=True)

1.5 Guardar datos limpios


In [34]:
import os

# Create the 'datos' directory if it doesn't exist
os.makedirs('../datos', exist_ok=True)

df_clean.to_csv("../datos/datos_limpios.csv", index=False)
print(f"Se procesaron {len(df_clean)} muestras válidas.")
df_clean.head()

Se procesaron 9935 muestras válidas.


,muestra,tiempo_min,T_guadua,H_guadua,T_plastico,H_plastico,T_ambiente,H_ambiente
0,1,0,1.0,0.7,2.0,-1.0,4.0,-2.1
1,2,1,1.0,24.5,2.0,24.1,4.0,20.2
2,3,2,1.0,24.1,2.0,24.5,4.0,20.9
3,4,3,1.0,24.1,2.0,24.6,4.0,20.7
4,1,0,1.0,24.2,2.0,25.2,4.0,20.3


In [36]:
display(df_clean.head(125))

,muestra,tiempo_min,T_guadua,H_guadua,T_plastico,H_plastico,T_ambiente,H_ambiente
0,1,0,1.0,0.7,2.0,-1.0,4.0,-2.1
1,2,1,1.0,24.5,2.0,24.1,4.0,20.2
2,3,2,1.0,24.1,2.0,24.5,4.0,20.9
3,4,3,1.0,24.1,2.0,24.6,4.0,20.7
4,1,0,1.0,24.2,2.0,25.2,4.0,20.3
...,...,...,...,...,...,...,...,...
120,116,115,1.0,20.6,2.0,21.2,4.0,16.9
121,117,116,1.0,20.5,2.0,21.1,4.0,16.3
122,118,117,1.0,20.6,2.0,21.4,4.0,16.3
123,119,118,1.0,20.6,2.0,21.6,4.0,16.9
